## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        '''网络结构：784 → 256 (ReLU) → 10'''
        ####################
        self.W1 = tf.Variable(
            tf.random.truncated_normal([28*28, 256], stddev=0.1), dtype=tf.float32)
        self.b1 = tf.Variable(tf.zeros([256]), dtype=tf.float32)
        
        self.W2 = tf.Variable(
            tf.random.truncated_normal([256, 10], stddev=0.1), dtype=tf.float32)
        self.b2 = tf.Variable(tf.zeros([10]), dtype=tf.float32)
    def __call__(self, x):
        x = tf.reshape(x, [-1, 28*28])
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.573483 ; accuracy 0.069016665
epoch 1 : loss 2.5480595 ; accuracy 0.0722
epoch 2 : loss 2.523996 ; accuracy 0.07645
epoch 3 : loss 2.5011058 ; accuracy 0.08116667
epoch 4 : loss 2.479237 ; accuracy 0.0856
epoch 5 : loss 2.4582682 ; accuracy 0.09053333
epoch 6 : loss 2.4380949 ; accuracy 0.09663333
epoch 7 : loss 2.4186287 ; accuracy 0.10275
epoch 8 : loss 2.399795 ; accuracy 0.1091
epoch 9 : loss 2.3815286 ; accuracy 0.11528333
epoch 10 : loss 2.363775 ; accuracy 0.12146667
epoch 11 : loss 2.3464906 ; accuracy 0.12803334
epoch 12 : loss 2.3296328 ; accuracy 0.1341
epoch 13 : loss 2.3131673 ; accuracy 0.14088333
epoch 14 : loss 2.297063 ; accuracy 0.14885
epoch 15 : loss 2.2812932 ; accuracy 0.1563
epoch 16 : loss 2.2658339 ; accuracy 0.1636
epoch 17 : loss 2.2506623 ; accuracy 0.17141667
epoch 18 : loss 2.23576 ; accuracy 0.17926666
epoch 19 : loss 2.2211099 ; accuracy 0.18728334
epoch 20 : loss 2.2066984 ; accuracy 0.19518334
epoch 21 : loss 2.192513 ; accuracy 0.2031